# OAPR COLAB_RUNNER — GPU jobs only

This notebook **only** runs GPU work. Model/code/configs live in the repo.

1. Mount Drive
2. Clone/pull repo
3. `pip install` requirements **without** mamba-ssm / causal-conv1d
4. Symlink COCO from Drive → `data/coco`
5. Run each `gpu_jobs/` script in its **own cell** (stop anytime if GPU is low)

Outputs → `/content/drive/MyDrive/oapr_results`

See `DRIVE_CHECKLIST.md` for exact Drive folders.

**Runtime:** Runtime → Change runtime type → GPU

In [ ]:
# Cell 0 — Mount Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cell 1 — Clone or pull repo
import os
from pathlib import Path

REPO_URL = os.environ.get(
    'OAPR_REPO_URL',
    'https://github.com/HamzaSattar9822/OAPR-Occlusion-Aware-Probabilistic-Reconstruction.git',
)
REPO_DIR = Path('/content/oapr_pose')

if REPO_DIR.exists():
    %cd {REPO_DIR}
    !git pull --ff-only || true
else:
    !git clone {REPO_URL} {REPO_DIR}
    %cd {REPO_DIR}

print('CWD:', Path.cwd())

In [ ]:
# Cell 2 — pip install requirements minus mamba / causal-conv
%cd /content/oapr_pose
!grep -v -E '^(mamba-ssm|causal-conv1d|#)' requirements.txt | grep -v '^$' > /tmp/reqs_nomamba.txt || true
!grep -v -E 'mamba-ssm|causal-conv1d' requirements.txt | sed '/^#/d' | sed '/^$/d' > /tmp/reqs_nomamba.txt
!pip install -q -r /tmp/reqs_nomamba.txt
!pip install -q thop
import torch
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
print('deps installed (mamba-ssm / causal-conv1d skipped → attention fallback)')

In [ ]:
# Cell 3 — Symlink COCO + create results dir
from pathlib import Path
import os

%cd /content/oapr_pose

coco_src = Path('/content/drive/MyDrive/coco')
coco_dst = Path('data/coco')
coco_dst.parent.mkdir(parents=True, exist_ok=True)
if coco_dst.exists() or coco_dst.is_symlink():
    if coco_dst.is_symlink() or coco_dst.is_file():
        coco_dst.unlink()
    else:
        raise SystemExit(f'Refuse to replace non-symlink {coco_dst}')
os.symlink(coco_src, coco_dst)

OUT = Path('/content/drive/MyDrive/oapr_results')
OUT.mkdir(parents=True, exist_ok=True)

CKPT = Path('/content/drive/MyDrive/oapr_checkpoints/best.pth')
CFG = 'configs/m3_oapr_complete.yaml'

assert coco_src.exists(), 'Missing Drive COCO at /content/drive/MyDrive/coco — see DRIVE_CHECKLIST.md'
assert (coco_src / 'images/val2017').is_dir(), 'Missing coco/images/val2017'
assert (coco_src / 'annotations/person_keypoints_val2017.json').is_file()
assert CKPT.is_file(), f'Missing checkpoint {CKPT}'

print('COCO symlink OK →', coco_dst.resolve())
print('Checkpoint:', CKPT)
print('Results →', OUT)

## GPU jobs (run **one cell at a time**)

Each cell writes under `/content/drive/MyDrive/oapr_results`. Stop between cells if GPU memory is low (Runtime → Restart runtime, remount Drive, re-run setup cells).

In [ ]:
# Cell 4 — 01_evaluate.py  (full COCO val AP / AP50 / APH)
%cd /content/oapr_pose
!python gpu_jobs/01_evaluate.py \
  --checkpoint /content/drive/MyDrive/oapr_checkpoints/best.pth \
  --config configs/m3_oapr_complete.yaml \
  --out /content/drive/MyDrive/oapr_results/01_evaluate.json \
  --override training.num_workers=2 training.batch_size=8

In [ ]:
# Cell 5 — 02_ablation.py  (full / no_gcn / no_mamba / no_confidence_gate)
%cd /content/oapr_pose
!python gpu_jobs/02_ablation.py \
  --checkpoint /content/drive/MyDrive/oapr_checkpoints/best.pth \
  --config configs/m3_oapr_complete.yaml \
  --out /content/drive/MyDrive/oapr_results/02_ablation.json \
  --override training.num_workers=2 training.batch_size=8

In [ ]:
# Cell 6 — 03_sensitivity.py  (sweep τ and β)
%cd /content/oapr_pose
!python gpu_jobs/03_sensitivity.py \
  --checkpoint /content/drive/MyDrive/oapr_checkpoints/best.pth \
  --config configs/m3_oapr_complete.yaml \
  --out /content/drive/MyDrive/oapr_results/03_sensitivity.json \
  --override training.num_workers=2 training.batch_size=8

In [ ]:
# Cell 7 — 04_complexity.py  (params / GFLOPs / peak GPU mem)
%cd /content/oapr_pose
!python gpu_jobs/04_complexity.py \
  --checkpoint /content/drive/MyDrive/oapr_checkpoints/best.pth \
  --config configs/m3_oapr_complete.yaml \
  --out /content/drive/MyDrive/oapr_results/04_complexity.json

In [ ]:
# Cell 8 — 05_residual_hist.py  (val residual histogram PNG)
%cd /content/oapr_pose
!python gpu_jobs/05_residual_hist.py \
  --checkpoint /content/drive/MyDrive/oapr_checkpoints/best.pth \
  --config configs/m3_oapr_complete.yaml \
  --out /content/drive/MyDrive/oapr_results/05_residual_hist.png \
  --override training.num_workers=2 training.batch_size=8

In [ ]:
# Cell 9 — 06_train.py (SLOW) — saves checkpoint under Drive results
# Tip: smoke-test first with --epochs 1
%cd /content/oapr_pose
!python gpu_jobs/06_train.py \
  --checkpoint /content/drive/MyDrive/oapr_checkpoints/best.pth \
  --config configs/m3_oapr_complete.yaml \
  --out /content/drive/MyDrive/oapr_results/06_train \
  --override training.num_workers=2 training.batch_size=8